# Hybridization

## Poisson equation

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(unit_square.GenerateMesh(maxh=0.2))

In [ ]:
order = 1

Sigma = HDiv(mesh, order=order)
V = L2(mesh, order=order-1)
Vhat = FacetFESpace(mesh, order=order, dirichlet=".*")

X = PrivateSpace(Sigma) * V * Vhat
(sigma, u, uhat), (tau, v, vhat) = X.TnT()

print ("ndof.Sigma=", Sigma.ndof, "V.ndof=", V.ndof, "Vhat.ndof=", Vhat.ndof)
print ("X.ndof = ", X.ndof)

In [ ]:
n = specialcf.normal(mesh.dim)
def Div(sigma, v, vhat): return div(sigma)*v*dx - sigma*n * vhat*dx(element_boundary=True)
source = x

lhs = sigma*tau*dx + Div(sigma,v,vhat) + Div(tau, u,uhat)
rhs = source*v*dx
gf = Solve(lhs==rhs)
_, gfu, gfuhat = gf.components
Draw (gfu);

## Reconstruct the flux

In [ ]:
Xp1 = Discontinuous(Sigma)*V
(sigmap1, up1), (taup1, vp1) = Xp1.TnT()

lhs = sigmap1*taup1*dx + div(sigmap1)*vp1*dx + div(taup1)*up1*dx
rhs = source*vp1*dx + taup1*n * gfuhat*dx(element_boundary=True)
gfpsigma,_ = Solve(lhs==rhs).components

Draw (gfpsigma[0], mesh);

## Postprocessing for the scalar

In [ ]:
Xp = L2(mesh,order=order+1) * L2(mesh,order=0)
(up,umean),(vp,vmean) = Xp.TnT()

lhs = grad(up)*grad(vp)*dx + up*vmean*dx + vp*umean*dx
rhs = gfpsigma*grad(vp)*dx + gfu*vmean*dx
gfp,_ = Solve(lhs==rhs).components
Draw (gfp);